# Models

A practical refresher on the **open-weight foundation models** you self-host in an inference/serving stack — the catalog side of MLOps. It focuses on the families you actually deploy on your own GPUs: **Llama 3 (8B / 70B)**, **Mistral 7B**, **Google Gemma 2 (9B / 27B-it)**, **OpenAI gpt-oss-20b**, and the encoder workhorse **BERT** — how to size them, quantize them, and choose between them.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

In an MLOps platform, a **model** is the actual set of trained weights you load onto a GPU and serve. This notebook is about the *open-weight* models you can download and run yourself (as opposed to closed APIs like GPT-4 or Claude). Picking the right one is the single decision that most affects your serving cost, latency, and quality.

### What is it?

An open-weight model is a checkpoint — typically a directory of `safetensors` shards plus a tokenizer and a `config.json` — published under a license that lets you run it on your own hardware. The families covered here:

- **Llama 3 8B / 70B** (Meta) — dense decoder-only transformers; the de-facto baselines for self-hosting. 8B fits a single 24 GB GPU; 70B needs multi-GPU or aggressive quantization.
- **Mistral 7B** (Mistral AI) — a fast, Apache-2.0 7B dense model using Grouped-Query Attention (GQA) and Sliding-Window Attention (SWA); excellent quality-per-byte.
- **Gemma 2 9B / 27B-it** (Google) — dense models with interleaved local/global attention; `-it` denotes the instruction-tuned variant you serve for chat.
- **gpt-oss-20b** (OpenAI, 2025) — a **Mixture-of-Experts** model (~21B total, ~3.6B active per token) shipped in MXFP4; runs in ~16 GB despite its total size.
- **BERT** (Google) — an *encoder-only* model. Not a chat model: you use it for embeddings, classification, retrieval reranking, and NER — cheap, fast, CPU-friendly.

### Why use it?

- **Data stays in your VPC** — no prompts leave your infrastructure.
- **Predictable cost** — you pay for GPU-hours, not per-token; high-throughput workloads get dramatically cheaper than API calls.
- **Control** — pin a version, fine-tune, quantize, and tune latency/throughput to your SLA.
- **No rate limits or vendor lock-in.**

### When to use it?

- High, steady request volume where per-token API pricing dominates cost.
- Strict data-residency / compliance requirements.
- Latency-sensitive paths where a small local model beats a network round-trip to an API.
- Embedding/classification pipelines (BERT-class) that run constantly and cheaply.

**When *not* to:** low/spiky volume where a managed API is cheaper than idle GPUs, or when you need frontier reasoning quality that today's open weights don't match — there, call a hosted API instead.

## Key Features

### What distinguishes these model families

| Model | Type | Params (total / active) | Context | License | Native fit (FP16) |
|-------|------|-------------------------|---------|---------|-------------------|
| Llama 3 8B | Dense decoder | 8B | 8K | Llama 3 Community | ~16 GB |
| Llama 3 70B | Dense decoder | 70B | 8K | Llama 3 Community | ~140 GB (multi-GPU) |
| Mistral 7B | Dense decoder (GQA+SWA) | 7.3B | 8K–32K | Apache 2.0 | ~15 GB |
| Gemma 2 9B | Dense decoder | 9B | 8K | Gemma | ~18 GB |
| Gemma 2 27B-it | Dense decoder (instr-tuned) | 27B | 8K | Gemma | ~54 GB (multi-GPU) |
| gpt-oss-20b | MoE decoder | 21B / 3.6B | 128K | Apache 2.0 | ~16 GB (MXFP4) |
| BERT (base) | Encoder | 110M | 512 | Apache 2.0 | <1 GB (CPU ok) |

Key levers that matter when serving:

- **Dense vs MoE** — dense models activate all weights every token (compute scales with size); MoE (gpt-oss) activates a few experts, so a 21B model has the *compute* of a ~4B model but the *memory* of a 21B one.
- **GQA (Grouped-Query Attention)** — fewer KV heads → smaller KV cache → more concurrent requests. All modern families here use it.
- **Instruction-tuned vs base** — serve the `-it`/`-Instruct` variant for chat; base models only do raw completion.
- **License** — Apache 2.0 (Mistral, gpt-oss, BERT) is the most permissive; Llama and Gemma have community licenses with acceptable-use terms and (for Llama) a >700M-MAU clause.

## Architecture Overview

All the generative models here are **decoder-only transformers** (next-token predictors); BERT is the odd one out — an **encoder** that sees the whole sequence bidirectionally. At serving time the lifecycle is the same regardless of family:

```
  Hugging Face Hub / S3 / model registry
        |  (safetensors shards + tokenizer + config.json)
        v
  Download & verify  ->  (optional) quantize: FP16 -> INT8 / INT4 / MXFP4
        |
        v
  Load onto GPU(s)   ->  weights in HBM + KV cache (grows with tokens x concurrency)
        |
   tensor / pipeline parallel across GPUs if it doesn't fit one card
        |
        v
  Inference server (vLLM / TGI / SGLang) -> batched decode -> tokens out
```

### Components

1. **Weights** (`*.safetensors`) — the parameters; the bulk of GPU memory. Size = params x bytes-per-param (2 for FP16, ~0.5 for INT4).
2. **Tokenizer** — maps text <-> token IDs; must match the model exactly (a mismatched tokenizer silently corrupts outputs).
3. **`config.json` / `generation_config.json`** — architecture dims, RoPE settings, and default sampling params.
4. **KV cache** — per-request memory holding past keys/values; often the real limit on concurrency, not the weights.
5. **Chat template** — the Jinja template (in the tokenizer) that formats messages into the exact prompt format the `-it` model was trained on. Getting this wrong is the #1 cause of "the open model is dumb."

## Installation

### Prerequisites

- Python 3.10+ and a CUDA-capable GPU (or CPU for BERT-class models).
- `transformers`, `torch`, `accelerate`; `huggingface_hub` for downloads.
- A Hugging Face account + access token, and **accepted license terms** on the model page for gated repos (Llama 3 and Gemma are gated; Mistral, gpt-oss, BERT are not).
- Enough disk and HBM for the chosen model (see the sizing table above).

### Installation Steps

**Note**: Uncomment the cell below to install the libraries (e.g. in Google Colab).

In [ ]:
# Uncomment to install the core open-model toolchain.
# !pip install -U "transformers>=4.44" torch accelerate huggingface_hub safetensors

# Authenticate once for gated repos (Llama 3, Gemma):
# from huggingface_hub import login
# login(token="hf_...")           # or: huggingface-cli login

# Pre-download a checkpoint to a local cache (recommended for reproducible deploys):
# huggingface-cli download mistralai/Mistral-7B-Instruct-v0.3 --local-dir ./mistral-7b
print("Install/auth steps shown above (commented so this notebook runs anywhere).")

## Basic Usage

### Sizing a model before you download it

Before pulling a 140 GB checkpoint, estimate whether it fits. GPU memory you need ≈ **weights + KV cache + overhead**. Weights = params × bytes-per-param; the KV cache grows with sequence length and concurrency. The helper below does the back-of-envelope math for any model in this notebook.

In [ ]:
# Estimate GPU memory for serving a given model at a given precision.
BYTES_PER_PARAM = {"fp16": 2.0, "bf16": 2.0, "int8": 1.0, "int4": 0.5, "mxfp4": 0.5}

MODELS = {
    # name: (total_params_B, active_params_B, hidden, n_layers, kv_heads, head_dim)
    "llama-3-8b":      (8.0,  8.0,  4096, 32,  8,  128),
    "llama-3-70b":     (70.0, 70.0, 8192, 80,  8,  128),
    "mistral-7b":      (7.3,  7.3,  4096, 32,  8,  128),
    "gemma-2-9b":      (9.2,  9.2,  3584, 42,  8,  256),
    "gemma-2-27b-it":  (27.2, 27.2, 4608, 46,  16, 128),
    "gpt-oss-20b":     (21.0, 3.6,  2880, 24,  8,  64),   # MoE: total vs active
    "bert-base":       (0.11, 0.11, 768,  12,  12, 64),
}

def weights_gb(name, dtype="fp16"):
    total = MODELS[name][0]
    return total * 1e9 * BYTES_PER_PARAM[dtype] / 1024**3

def kv_cache_gb(name, seq_len, batch, dtype="fp16"):
    _, _, _, n_layers, kv_heads, head_dim = MODELS[name]
    # 2 (K and V) * layers * kv_heads * head_dim * seq_len * batch * bytes
    bytes_ = 2 * n_layers * kv_heads * head_dim * seq_len * batch * BYTES_PER_PARAM[dtype]
    return bytes_ / 1024**3

print(f"{'model':16s} {'fp16':>8s} {'int8':>8s} {'int4':>8s}")
for m in MODELS:
    print(f"{m:16s} {weights_gb(m,'fp16'):7.1f}G {weights_gb(m,'int8'):7.1f}G {weights_gb(m,'int4'):7.1f}G")

print("\nKV cache for 100 concurrent requests @ 4k tokens (fp16):")
for m in ["llama-3-8b", "llama-3-70b", "mistral-7b", "gemma-2-27b-it"]:
    print(f"  {m:16s} {kv_cache_gb(m, 4096, 100):6.1f} GB")

### Loading and running a model

The Transformers API is identical across the generative families — only the model ID and the chat template differ. For an instruction-tuned model, **always** format the prompt with `apply_chat_template` so it matches training.

In [ ]:
# Generative model (decoder) — runs as-is if you have the weights + a GPU.
# Swap MODEL_ID for any of: meta-llama/Meta-Llama-3-8B-Instruct,
# mistralai/Mistral-7B-Instruct-v0.3, google/gemma-2-9b-it, openai/gpt-oss-20b
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

def chat_once(model_id, user_msg, max_new_tokens=128):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="auto",
    )
    # The chat template formats messages into the exact prompt the -it model expects.
    inputs = tok.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)

# Guarded so the notebook runs without a GPU/weights present.
try:
    print(chat_once(MODEL_ID, "Explain KV cache in one sentence."))
except Exception as e:
    print(f"(Skipping live load — needs the weights + a GPU.) {type(e).__name__}: {e}")

## Advanced Features

### Quantization, MoE, and long context

#### Quantization — fit bigger models on smaller GPUs

Quantization stores weights in fewer bits. The quality loss from 4-bit is usually small for inference, and it roughly **quarters** the memory vs FP16:

- **INT8 / FP8** — ~half the memory, near-lossless; FP8 is hardware-accelerated on H100/Ada.
- **INT4** (GPTQ, AWQ, bitsandbytes NF4) — ~quarter the memory; lets a 70B run on a single 48 GB GPU.
- **MXFP4** — the 4-bit micro-scaled format gpt-oss ships in natively, so the 21B MoE loads in ~16 GB.

#### Mixture-of-Experts (gpt-oss-20b)

An MoE layer has many expert MLPs but a router activates only the top-k per token. gpt-oss-20b has ~21B total parameters but only ~3.6B active, so its **compute** (and latency) is like a small model while its **memory** is like a 21B model. Great throughput-per-FLOP; you still pay the full memory bill.

#### Long context

Context length is bounded by training (RoPE settings in `config.json`) and by **KV-cache memory** at serve time. gpt-oss-20b supports 128K; the Llama/Gemma/Mistral families here are 8K–32K. Doubling context doubles KV-cache memory per request — often the real concurrency limit.

In [ ]:
# Load any of these models in 4-bit to cut memory ~4x (needs bitsandbytes + GPU).
def load_4bit(model_id):
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",            # NormalFloat4: best quality 4-bit
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,       # quantize the quant constants too
    )
    return AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map="auto",
    )

# Memory saving illustrated with the sizing helper from earlier:
for m in ["llama-3-70b", "gemma-2-27b-it"]:
    print(f"{m}: fp16 {weights_gb(m,'fp16'):.0f}G -> int4 {weights_gb(m,'int4'):.0f}G")
print("\n(load_4bit(...) performs the real load when a GPU + bitsandbytes are present.)")

## Use Cases

### Real-world applications, mapped to the right model

#### Use Case 1: Internal chat assistant / RAG

- **Context**: Q&A over private docs for thousands of employees; data must stay in-VPC.
- **Implementation**: Serve **Llama 3 8B-Instruct** or **Mistral 7B-Instruct** on a single A10G/L4; retrieve with a BERT-class embedder; scale replicas behind a load balancer.
- **Results**: Sub-second responses at a fraction of per-token API cost; no data egress.

#### Use Case 2: High-quality summarization / drafting

- **Context**: Quality matters more than latency; budget allows multi-GPU.
- **Implementation**: **Llama 3 70B** or **Gemma 2 27B-it** with tensor parallelism across 2–4 GPUs, or INT4 on a single 48–80 GB card.
- **Results**: Near-frontier quality on-prem; cost amortized across steady volume.

#### Use Case 3: Cost-efficient agentic / tool-use workloads

- **Context**: Many short reasoning steps; throughput-bound.
- **Implementation**: **gpt-oss-20b** (MoE) for high tokens/sec per GPU-dollar at long (128K) context.
- **Results**: Strong reasoning at small-model latency thanks to sparse activation.

#### Use Case 4: Embeddings, classification, reranking

- **Context**: Search, dedup, moderation, NER — constant high volume, no generation needed.
- **Implementation**: **BERT** (or a sentence-transformer variant) on CPU or a cheap GPU; batch aggressively.
- **Results**: Millions of docs/day at trivial cost; no LLM required.

## Best Practices

### Recommended practices for serving open-weight models

1. **Pick the smallest model that meets your quality bar.** Test 7–9B first; only move to 27B/70B if evals demand it. Most of the cost is the model you chose.
2. **Always use the chat template** (`apply_chat_template`) and the `-it`/`-Instruct` variant for chat. A wrong prompt format makes a good model look broken.
3. **Quantize deliberately.** INT8/FP8 is near-lossless; jump to INT4/MXFP4 to fit a bigger model on one GPU — and re-run your evals after quantizing.
4. **Pin exact versions** (model revision/commit hash + tokenizer) and mirror weights to your own S3/registry. Don't depend on the Hub being up at deploy time.
5. **Size for the KV cache, not just the weights.** Concurrency × context length drives memory; budget for it explicitly (use the helper above).
6. **Use a real inference server** (vLLM / TGI / SGLang) with continuous batching and PagedAttention rather than raw `model.generate` in production.

## Common Pitfalls

### What to avoid

1. **Skipping the chat template** — feeding raw text to an instruction-tuned model. Symptoms: rambling, ignored instructions. Always format messages with the tokenizer's template.
2. **Forgetting license/gating terms** — Llama 3 and Gemma are gated and have acceptable-use clauses (Llama adds a >700M-MAU restriction). Confirm your use is allowed before shipping.
3. **Underestimating the KV cache** — the weights fit but you OOM under load. The cache scales with concurrency × sequence length; size for peak.
4. **Quantizing without re-evaluating** — assuming INT4 is free. It's usually fine, but verify on *your* task; some workloads are sensitive.
5. **Using BERT for generation (or an LLM for embeddings) by default** — BERT can't generate text; a 7B LLM is wasteful for pure classification. Match the model class to the job.
6. **Tokenizer/model mismatch** — loading weights from one repo with a tokenizer from another silently corrupts output. Load both from the same revision.

## Performance Optimization

### Getting more tokens/sec per GPU

Levers, roughly in order of impact:

- **Continuous batching + PagedAttention** (vLLM/TGI/SGLang) — the biggest throughput win; pack many requests into each forward pass with no KV-cache fragmentation.
- **Quantization** (FP8/INT8/INT4/MXFP4) — smaller weights → more KV-cache headroom → higher concurrency, and faster memory-bound decode.
- **Tensor parallelism** — split a 70B/27B across GPUs to fit and to lower latency; keep TP within one node (use fast NVLink, not the network).
- **GQA / smaller KV cache** — already built into these models; choosing a GQA model directly raises max concurrency.
- **Right precision for the hardware** — BF16 on A100/H100; FP8 on H100/Ada for free speedups.
- **Speculative decoding** — pair a big model with a tiny draft model to cut latency on easy tokens.

The cell below shows the kind of estimate you'd use to choose a GPU: does the model + KV cache for your target concurrency fit?

In [ ]:
# Capacity check: will (weights @ dtype) + (KV cache for N requests) fit a GPU?
def fits_gpu(name, gpu_gb, dtype, concurrency, seq_len, overhead_gb=2.0):
    w = weights_gb(name, dtype)
    kv = kv_cache_gb(name, seq_len, concurrency)
    total = w + kv + overhead_gb
    ok = total <= gpu_gb
    print(f"{name:16s} {dtype:6s} on {gpu_gb:>3.0f}GB GPU | "
          f"weights {w:5.1f} + kv {kv:5.1f} + oh {overhead_gb:.0f} = {total:6.1f}GB "
          f"-> {'FITS' if ok else 'OOM'} @ {concurrency} reqs x {seq_len} tok")
    return ok

fits_gpu("llama-3-8b",     24, "fp16", concurrency=16, seq_len=4096)   # A10G/L4
fits_gpu("mistral-7b",     24, "fp16", concurrency=32, seq_len=2048)
fits_gpu("gemma-2-27b-it", 48, "int4", concurrency=16, seq_len=4096)   # single L40S/A6000
fits_gpu("llama-3-70b",    80, "int4", concurrency=32, seq_len=4096)   # single H100
fits_gpu("llama-3-70b",    80, "fp16", concurrency=32, seq_len=4096)   # needs multi-GPU

## Production Deployment

### Deploying open-weight models

In practice you don't write your own generation loop — you run a dedicated inference server that gives you OpenAI-compatible APIs, continuous batching, and tensor parallelism out of the box.

#### vLLM (single command)

```bash
# Serve any HF model with an OpenAI-compatible API on :8000
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Meta-Llama-3-8B-Instruct \
  --dtype bfloat16 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.90        # leave headroom for the KV cache

# 70B across 4 GPUs with tensor parallelism + 4-bit weights:
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Meta-Llama-3-70B-Instruct \
  --tensor-parallel-size 4 --quantization awq
```

#### Docker (Text Generation Inference)

```dockerfile
FROM ghcr.io/huggingface/text-generation-inference:latest
ENV MODEL_ID=mistralai/Mistral-7B-Instruct-v0.3
ENV QUANTIZE=bitsandbytes-nf4
# Run with the GPU and the HF token mounted:
#   docker run --gpus all -e HF_TOKEN=$HF_TOKEN -p 8080:80 <image>
```

#### Kubernetes (GPU pod serving vLLM)

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: llama3-8b
spec:
  replicas: 2
  selector: { matchLabels: { app: llama3-8b } }
  template:
    metadata: { labels: { app: llama3-8b } }
    spec:
      containers:
        - name: vllm
          image: vllm/vllm-openai:latest
          args: ["--model", "meta-llama/Meta-Llama-3-8B-Instruct",
                 "--gpu-memory-utilization", "0.9"]
          ports: [{ containerPort: 8000 }]
          resources:
            limits:
              nvidia.com/gpu: 1
          env:
            - name: HF_TOKEN
              valueFrom: { secretKeyRef: { name: hf, key: token } }
```

> Tip: mirror weights into your own S3/registry and set `HF_HUB_OFFLINE=1` so pods don't depend on the Hub at startup.

## Monitoring and Observability

### Monitoring model serving in production

#### Key metrics to track

- **Time-to-first-token (TTFT)** — prefill latency; what users feel first.
- **Inter-token latency / tokens-per-second** — decode speed; sets perceived "typing" speed.
- **Throughput (requests & tokens/sec)** — capacity per replica; drives cost.
- **KV-cache utilization & queue depth** — the leading indicator of saturation; when the cache fills, new requests queue or get preempted.
- **GPU utilization & HBM usage** — are you compute-bound or memory-bound?
- **Output-quality signals** — refusal rate, truncation (hit `max_tokens`), and offline eval scores to catch a bad model/quantization swap.

#### Logging best practices

- Log the **model ID + revision/commit hash + quantization** with every request so a regression is traceable to a checkpoint change.
- Sample prompts/outputs (with PII controls) for offline evals and red-teaming.
- Export vLLM/TGI Prometheus metrics (they ship a `/metrics` endpoint) to Grafana; alarm on TTFT p99 and KV-cache-full events.
- Track **cost per 1M tokens** per model so you can compare self-hosting vs an API honestly.

## Troubleshooting

### Common issues

#### Issue 1: CUDA out of memory on load or under load

**Symptoms**: OOM at startup, or fine when idle but crashes under concurrency.

**Cause**: Weights too large for the GPU, or the KV cache grew with concurrency/context.

**Solution**: Quantize (INT4/AWQ/MXFP4), enable tensor parallelism, lower `--max-model-len` or `--gpu-memory-utilization`, or cap concurrency. Use the sizing helper above to right-size first.

#### Issue 2: The model gives rambling or low-quality answers

**Symptoms**: Ignores instructions, repeats, or replies oddly despite being a strong model.

**Cause**: Wrong prompt format (no chat template), serving the *base* instead of the `-it`/`-Instruct` variant, or a tokenizer mismatch.

**Solution**: Use `apply_chat_template`, load the instruction-tuned repo, and load tokenizer + weights from the same revision.

#### Issue 3: `GatedRepoError` / 403 when downloading

**Symptoms**: Download fails for Llama 3 or Gemma.

**Cause**: License not accepted, or no/invalid HF token.

**Solution**: Accept the terms on the model's Hub page, `huggingface-cli login`, and ensure the token (or `HF_TOKEN`) is available to the serving pod.

#### Issue 4: Quantized model is noticeably worse

**Symptoms**: Eval scores drop after switching to INT4.

**Cause**: Aggressive quantization on a sensitive task.

**Solution**: Move to INT8/FP8, use a better 4-bit method (AWQ/GPTQ over naive RTN), or keep FP16 with tensor parallelism. Always re-run evals after quantizing.

## Comparison with Alternatives

### Choosing among the families (and vs closed APIs)

| Dimension | Llama 3 8B | Mistral 7B | Gemma 2 9B | Llama 3 70B / Gemma 2 27B | gpt-oss-20b | BERT | Closed API (GPT-4/Claude) |
|-----------|-----------|-----------|-----------|---------------------------|-------------|------|---------------------------|
| Type | Dense LLM | Dense LLM | Dense LLM | Dense LLM (big) | MoE LLM | Encoder | Hosted LLM |
| Self-host fit | 1x 24GB | 1x 24GB | 1x 24GB | multi-GPU / INT4 | 1x 16GB | CPU/small GPU | n/a |
| Quality | Good | Good | Good | Best (open) | Strong reasoning | n/a (no gen) | Frontier |
| Cost model | GPU-hours | GPU-hours | GPU-hours | GPU-hours (more) | GPU-hours | trivial | per-token |
| License | Llama Community | Apache 2.0 | Gemma | Llama / Gemma | Apache 2.0 | Apache 2.0 | Terms of service |
| Best for | General chat/RAG | Fast, permissive | Google-tuned chat | Top open quality | Throughput + 128K | Embeddings/classify | Max quality, low volume |

### When to choose which

- **Start with Mistral 7B or Llama 3 8B** for general self-hosted chat/RAG — cheapest path that's usually good enough.
- **Gemma 2 9B** when you want a strong, instruction-tuned alternative in the same size class.
- **Llama 3 70B / Gemma 2 27B-it** when evals demand higher quality and you can afford multi-GPU or INT4.
- **gpt-oss-20b** for throughput-sensitive, long-context, or reasoning-heavy workloads where MoE efficiency wins.
- **BERT** for embeddings, classification, reranking, and NER — never for generation.
- **A closed API** when volume is low/spiky or you need frontier quality that open weights don't yet match.

## Resources

### Official Documentation & Model Cards

- Llama 3 (Meta): https://ai.meta.com/blog/meta-llama-3/ and https://huggingface.co/meta-llama
- Mistral 7B: https://mistral.ai/news/announcing-mistral-7b/ and https://huggingface.co/mistralai
- Gemma 2 (Google): https://ai.google.dev/gemma and https://huggingface.co/google
- gpt-oss (OpenAI): https://openai.com/index/introducing-gpt-oss/ and https://huggingface.co/openai/gpt-oss-20b
- BERT: https://arxiv.org/abs/1810.04805 and https://huggingface.co/google-bert/bert-base-uncased

### Tooling & Guides

- Hugging Face Transformers docs: https://huggingface.co/docs/transformers
- vLLM (continuous batching, PagedAttention): https://docs.vllm.ai
- Text Generation Inference (TGI): https://huggingface.co/docs/text-generation-inference
- SGLang: https://github.com/sgl-project/sglang
- Quantization (bitsandbytes / AWQ / GPTQ): https://huggingface.co/docs/transformers/quantization

### Community Resources

- Open LLM Leaderboard: https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard
- LMSYS Chatbot Arena: https://lmarena.ai
- r/LocalLLaMA: https://www.reddit.com/r/LocalLLaMA/

### Related Technologies

- KV cache & PagedAttention — the memory structure that limits serving concurrency
- Tensor / pipeline parallelism — how big models span multiple GPUs
- LoRA / QLoRA — parameter-efficient fine-tuning of these base models